In [1]:
import torch
from torch import nn
from tqdm import tqdm
import copy
from torch.utils.data import Subset
import torch.nn.functional as F

import sys, os
sys.path.append(os.path.abspath("../"))
import preprocessing

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

# Base MNIST NN

class NinaProClassificationModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv1d(10, 64, kernel_size=5, stride=2)
    self.relu1 = nn.ReLU()
    self.pool1 = nn.MaxPool1d(kernel_size=2)

    self.conv2 = nn.Conv1d(64, 128, kernel_size=5, stride=2)
    self.relu2 = nn.ReLU()
    self.pool2 = nn.MaxPool1d(kernel_size=2)

    self.flatten = nn.Flatten()
    
    self.dense1 = nn.LazyLinear(256)
    self.relu3 = nn.ReLU()
    self.dense2 = nn.Linear(256, 128)
    self.relu4 = nn.ReLU()
    self.dense3 = nn.Linear(128, 13)

  def forward(self, x):
    out = self.relu1(self.conv1(x))
    out = self.pool1(out)
    out = self.relu2(self.conv2(out))
    out = self.pool2(out)
    out = self.flatten(out)
    out = self.relu3(self.dense1(out))
    out = self.relu4(self.dense2(out))
    out = self.dense3(out)
    return out

class NinaProDataset(torch.utils.data.Dataset):
  def __init__(self, x, y):
    self.X = torch.tensor(x, dtype=torch.float32)
    self.y = torch.tensor(y, dtype=torch.long)

  def __len__(self):
    return len(self.X)
  
  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

# Base NN


model = NinaProClassificationModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

x_train, y_train, x_test, y_test, class_weights = preprocessing.multi_preprocess(exercise_number=1, path="../NinaProData")
num_batches = 100

train_data = NinaProDataset(x_train, y_train)
test_data = NinaProDataset(x_test, y_test)
batched_train = torch.utils.data.DataLoader(train_data, batch_size=num_batches, shuffle=True)
batched_test = torch.utils.data.DataLoader(test_data, batch_size=num_batches, shuffle=False)

loss_func = nn.CrossEntropyLoss()

Using cpu device
subject #[[1]]
exercise #[[1]]
0.0     351
5.0      58
3.0      53
1.0      50
8.0      44
6.0      43
12.0     42
4.0      41
10.0     39
2.0      37
7.0      36
9.0      35
11.0     35
Name: count, dtype: int64
0.0     908
3.0      30
1.0      26
10.0     25
12.0     23
5.0      22
4.0      18
6.0      17
7.0      17
9.0      17
2.0      16
8.0      16
11.0     15
Name: count, dtype: int64
subject #[[2]]
exercise #[[1]]
0.0     291
5.0      55
7.0      54
3.0      53
8.0      50
11.0     50
4.0      49
2.0      48
9.0      48
6.0      46
12.0     42
10.0     39
1.0      37
Name: count, dtype: int64
0.0     880
8.0      28
3.0      26
1.0      24
5.0      24
4.0      23
7.0      22
9.0      21
10.0     21
2.0      20
6.0      20
11.0     18
12.0     18
Name: count, dtype: int64
subject #[[3]]
exercise #[[1]]
0.0     339
2.0      54
3.0      52
5.0      47
6.0      47
12.0     46
1.0      44
7.0      43
4.0      42
8.0      40
9.0      39
10.0     35
11.0     34
Name: 